<a href="https://colab.research.google.com/github/gandersen-ctb/skilljar/blob/main/Prompt_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Promp Eval workflow

In [1]:
# install deps.
%pip install anthropic python-dotenv

In [2]:
from shared import max_tokens, add_role_message, chat, run_eval

In [3]:
import json

def generate_test_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    messages = []
    add_role_message(messages, 'user', prompt)
    add_role_message(messages, 'assistant', "```json")
    model = "claude-haiku-4-5"
    text = chat(messages, model, stop_sequences=["```"])

    return json.loads(text)

In [4]:
dataset = generate_test_dataset()
dataset

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket ARN (e.g., 'arn:aws:s3:::my-bucket'). Return 'us-east-1' as the default if no region is specified in the ARN."},
 {'task': "Create a JSON object representing an AWS IAM policy that grants read-only access to all objects in an S3 bucket named 'data-lake'."},
 {'task': "Write a regular expression that validates an AWS EC2 instance ID format (e.g., 'i-0a1b2c3d4e5f6g7h8')."}]

# Evaluate the dateset

In [5]:
results = run_eval(dataset);
print(json.dumps(results, indent=2))

Average score: 7
[
  {
    "output": "# AWS Region Extractor from S3 Bucket ARN\n\n## Solution\n\n```python\ndef extract_region_from_s3_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket ARN.\n    \n    ARN format: arn:partition:service:region:account-id:resource\n    Example: 'arn:aws:s3:::my-bucket' (no region specified)\n             'arn:aws:s3:us-west-2::my-bucket' (region specified)\n    \n    Args:\n        arn: The S3 bucket ARN string\n        \n    Returns:\n        The AWS region if specified, otherwise 'us-east-1' as default\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    DEFAULT_REGION = \"us-east-1\"\n    \n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    parts = arn.split(\":\")\n    \n    # Valid ARN must have at least 6 parts: arn:partition:service:region:account:resource\n    if len(parts) < 6:\n        raise ValueError(f\"Invalid